# Convert raw data to 'strict' json

In [1]:
import json
import gzip
import os
dataset_name = "Beauty"
os.makedirs(dataset_name, exist_ok=True)

def parse(path):
  g = gzip.open(path, 'r')
  for l in g:
    yield json.dumps(eval(l))

# Beauty dataset
f = open(f"./{dataset_name}/{dataset_name}.json", 'w')
for l in parse(f"reviews_{dataset_name}_5.json.gz"):
  f.write(l + '\n')

In [2]:
# print the number of lines in the file and the first line
data = open(f"./{dataset_name}/{dataset_name}.json", 'r')
print("Number of lines:", sum(1 for _ in data))
data.seek(0)  # Reset file pointer to the beginning
print("First line:", data.readline().strip())
data.close()

Number of lines: 198495
First line: {"reviewerID": "A1YJEY40YUW4SE", "asin": "7806397051", "reviewerName": "Andrea", "helpful": [3, 4], "reviewText": "Very oily and creamy. Not at all what I expected... ordered this to try to highlight and contour and it just looked awful!!! Plus, took FOREVER to arrive.", "overall": 1.0, "summary": "Don't waste your money", "unixReviewTime": 1391040000, "reviewTime": "01 30, 2014"}


In [4]:
import numpy as np
import pandas as pd
import json

# Initialize mapping dictionaries
userID_mapping = {}
itemID_mapping = {}

# Open the JSON file for reading
data = open(f"./{dataset_name}/{dataset_name}.json", 'r')

# Initialize lists to store userID, itemID, and timestamp
userIDs = []
itemIDs = []
timestamps = []

# Process each line in the JSON file
for line in data:
    review = json.loads(line.strip())
    userID = review['reviewerID']
    itemID = review['asin']
    timestamp = review['unixReviewTime']
    
    # Map userID to an integer starting from 1
    if userID not in userID_mapping:
        userID_mapping[userID] = len(userID_mapping) + 1
    
    # Map itemID to an integer starting from 1
    if itemID not in itemID_mapping:
        itemID_mapping[itemID] = len(itemID_mapping) + 1
    
    # Append mapped values and timestamp to lists
    userIDs.append(userID_mapping[userID])
    itemIDs.append(itemID_mapping[itemID])
    timestamps.append(timestamp)

# Save mapping dictionaries as .npy files
np.save(f'./{dataset_name}/user_mapping.npy', userID_mapping)
print("user_num:", len(userID_mapping))
print("the first five userID mapping:", list(userID_mapping.items())[:5])
np.save(f'./{dataset_name}/item_mapping.npy', itemID_mapping)
print("item_num:", len(itemID_mapping))
print("the first five itemID mapping:", list(itemID_mapping.items())[:5])

# Group itemIDs by userID and sort by timestamp
user_item_mapping = {}
for userID, itemID, timestamp in zip(userIDs, itemIDs, timestamps):
    if userID not in user_item_mapping:
        user_item_mapping[userID] = []
    user_item_mapping[userID].append((itemID, timestamp))

# Sort itemIDs for each user by timestamp
for userID in user_item_mapping:
    user_item_mapping[userID].sort(key=lambda x: x[1])
    user_item_mapping[userID] = [item[0] for item in user_item_mapping[userID]]

# Print a sample of the results
print("user-item mapping:", list(user_item_mapping.items())[:5])

# Split data into training, validation, and testing sets using leave-one-out strategy
train_data = {}
val_data = {}
test_data = {}

for userID, item_sequence in user_item_mapping.items():
    # Assign the last item for testing, the second-to-last for validation, and the rest for training
    train_data[userID] = item_sequence[:-2]
    val_data[userID] = item_sequence[:-1]
    test_data[userID] = item_sequence

# Print a sample of the split data
# print("training data:", list(train_data.items())[:5])
# print("validation data:", list(val_data.items())[:5])
# print("testing data:", list(test_data.items())[:5])

# Prepare data for train, validation, and test sets
def prepare_data(data_dict):
    rows = []
    for userID, item_sequence in data_dict.items():
        history = item_sequence[:-1]
        target = item_sequence[-1]
        rows.append({'user': userID, 'history': history, 'target': target})
    return pd.DataFrame(rows)

# Create dataframes for train, validation, and test sets
train_df = prepare_data(train_data)
print("\nTraining data shape:", train_df.shape)
print("the first 3 rows of training data:\n", train_df.head(3))
val_df = prepare_data(val_data)
print("\nValidation data shape:", val_df.shape)
print("the first 3 rows of validation data:\n", val_df.head(3))
test_df = prepare_data(test_data)
print("\nTesting data shape:", test_df.shape)
print("the first 3 rows of testing data:\n", test_df.head(3))

# Save dataframes to parquet files
train_df.to_parquet(f'./{dataset_name}/train.parquet', index=False)
val_df.to_parquet(f'./{dataset_name}/valid.parquet', index=False)
test_df.to_parquet(f'./{dataset_name}/test.parquet', index=False)

print("Data saved to parquet files.")

data.close()


user_num: 22363
the first five userID mapping: [('A1YJEY40YUW4SE', 1), ('A60XNB876KYML', 2), ('A3G6XNM240RMWA', 3), ('A1PQFP6SAJ6D80', 4), ('A38FVHZTNQ271F', 5)]
item_num: 12101
the first five itemID mapping: [('7806397051', 1), ('9759091062', 2), ('9788072216', 3), ('9790790961', 4), ('9790794231', 5)]
user-item mapping: [(1, [6846, 7873, 4585, 1, 5406]), (2, [816, 10406, 11194, 11651, 9716, 1, 233]), (3, [1, 6050, 7977, 5252, 4211, 243, 11204, 5863, 6609]), (4, [5522, 439, 5161, 11140, 1, 7849]), (5, [1, 10470, 10064, 9403, 10362, 4758, 6500, 11444, 11390])]

Training data shape: (22363, 3)
the first 3 rows of training data:
    user                           history  target
0     1                      [6846, 7873]    4585
1     2        [816, 10406, 11194, 11651]    9716
2     3  [1, 6050, 7977, 5252, 4211, 243]   11204

Validation data shape: (22363, 3)
the first 3 rows of validation data:
    user                                  history  target
0     1                       [684

# Generate Item Semantic Embeddings

In [4]:
# Beauty metadata 
f = open(f"./{dataset_name}/{dataset_name}_metadata.json", 'w')
for l in parse(f"meta_{dataset_name}.json.gz"):
  f.write(l + '\n')

In [8]:
# Open the metadata file for reading
with open(f"./{dataset_name}/{dataset_name}_metadata.json", 'r') as metadata_file:
    # Create a reverse mapping from itemID to asin
    reverse_itemID_mapping = {v: k for k, v in itemID_mapping.items()}
    
    # Initialize a dictionary to store the extracted information
    item_info = {}
    
    # Process each line in the metadata file
    for line in metadata_file:
        metadata = json.loads(line.strip())
        asin = metadata.get('asin')
        
        # Check if the asin exists in the reverse mapping
        if asin in reverse_itemID_mapping.values():
            itemID = itemID_mapping[asin]
            item_info[itemID] = {
                'title': metadata.get('title') if metadata.get('title') else None,
                'price': metadata.get('price') if metadata.get('price') else None,
                'salesRank': metadata.get('salesRank') if metadata.get('salesRank') else None,
                'brand': metadata.get('brand') if metadata.get('brand') else None,
                'categories': metadata.get('categories') if metadata.get('categories') else None,
            }

# Print the information for the first 5 items
for itemID, info in list(item_info.items())[:5]:
    print(f"ItemID: {itemID}, Info: {info}")

KeyError: 'Beauty'

In [17]:
# 👈👈👈
dataset_name = "Beauty"
# Open the metadata file for reading

set_c = set()
set_r = set()
with open(f"./{dataset_name}/{dataset_name}_metadata.json", 'r') as metadata_file:
    # Create a reverse mapping from itemID to asin
    reverse_itemID_mapping = {v: k for k, v in itemID_mapping.items()}

    # Process each line in the metadata file
    for line in metadata_file:
        metadata = json.loads(line.strip())
        asin = metadata.get('asin')

        # Check if the asin exists in the reverse mapping
        if asin in reverse_itemID_mapping.values():
            c = metadata.get('categories')[0] if metadata.get('categories') else None
            for cc in c:
                set_c.add(cc)

            r = metadata.get('salesRank') if metadata.get('salesRank') else None
            if r:
                for rr in r:
                    set_r.add(rr)

print(set_c)
print(set_r)

{'Chemical Hair Dyes', 'Cosmetic Bags', 'Ridge Filler', 'Hair Cutting Kits', 'Cuticle Pushers', 'Travel Cases & Holders', "Women's", 'Nail Whitening', 'Complexes', 'Bath Pillows', 'Eye Masks', 'Fillers', 'Oils', 'Rollers & Pens', 'Body Souffles & Mousse', 'Acids & Peels', 'Bronzers & Highlighters', 'Makeup', 'Irons', 'Styling Products', 'Beauty', 'Bars', 'Hair Coloring Tools', 'Body Glitter', 'Nails', 'Top & Base Coats', 'Bath Brushes', 'Cotton Swabs', 'Scissors', 'Fragrance', 'Makeup Remover', 'Ponytail Holders', 'Lipstick Primers', 'Nail Dryers', 'Eyeliner', 'Combs', 'Sun', 'Body', 'Sharpeners', 'Diffusers', 'Hair Relaxers', 'Braid Maintenance', 'Lips', 'Eyes', 'Sets & Kits', 'Color Refreshers', 'Powder', 'Bath & Body', 'Bath Mitts & Cloths', 'Tote Bags', 'Applicator Bottles', 'Hot-Air Brushes', 'Claws', 'Hair & Scalp Treatments', 'Blush', 'Home Permanent Kits', 'Masks & Pillows', 'Nail Art Equipment', 'Curlers', 'Hands & Nails', 'Cloths & Towelettes', 'Eye Shadow', 'Hair Dryers', 'M

In [22]:
# 👈👈👈
dataset_name = "Beauty"
# Open the metadata file for reading

with open(f"./{dataset_name}/{dataset_name}_metadata.json", 'r') as metadata_file:
    # Create a reverse mapping from itemID to asin
    reverse_itemID_mapping = {v: k for k, v in itemID_mapping.items()}

    # Initialize a dictionary to store the extracted information
    item_info = {}

    # Process each line in the metadata file
    for line in metadata_file:
        metadata = json.loads(line.strip())
        asin = metadata.get('asin')

        # Check if the asin exists in the reverse mapping
        if asin in reverse_itemID_mapping.values():
            itemID = itemID_mapping[asin]
            item_info[itemID] = {
                # 1. text
                # 'xxxxxxx'
                'title': metadata.get('title') if metadata.get('title') else None,
                # {'Beauty': 10486, "xxx": xxx}
                'salesRank': metadata.get('salesRank') if metadata.get('salesRank') else None,  # 不太好处理
                # 'xxxxxxxxxxxxxxxx'
                'description': metadata.get('description') if metadata.get('description') else None,

                # 2. image
                # 'imUrl': 'http://ecx.images-amazon.com/images/I/41Rn18OeU6L._SY300_.jpg'
                'imUrl': metadata.get('imUrl') if metadata.get('imUrl') else None,

                # 3. num & cls
                # 190
                'price': metadata.get('price') if metadata.get('price') else None,
                # 'COKA'
                'brand': metadata.get('brand') if metadata.get('brand') else None,
                # ['Beauty', 'Makeup', 'Face', 'Concealers & Neutralizers']
                'categories': metadata.get('categories')[0] if metadata.get('categories') else None,
            }
        # asin = metadata.get('asin')
        #
        # # Check if the asin exists in the reverse mapping
        # if asin in reverse_itemID_mapping.values():
        #     # for k in metadata.keys():
        #     #     print(k, ": ", metadata[k])
        #     # break

# Print the information for the first 5 items
for itemID, info in list(item_info.items())[:5]:
    print(f"ItemID: {itemID}, Info: {info}")

ItemID: 1, Info: {'title': 'WAWO 15 Color Professionl Makeup Eyeshadow Camouflage Facial Concealer Neutral Palette', 'salesRank': {'Beauty': 10486}, 'description': 'An extensive range of 15 multiple vibrant long wear concealer colour with different skin tones to create more than 10,000 amazing looks. Using the most commonly applied shades, ensures the best skin colour match and guarantees a traceless and natural finish. Enabling layering and mixing, provides total camouflage for almost any skin problem including blemishes, scars, birthmarks and black circles. It is also suitable to use as bronzer. The light colour is suitable for redness, acne and so on. The medium colour is perfect for dark shadows in the under-eye area. The dark colour provides exceptional camouflage and adheres well to the skin. Silky glossy colour and high quality ingredients together to care skin around and can last for all day long. It is perfect for Professional Salon, Wedding, Party and Home use. Size: 15.4 x 1

In [27]:
# Prepare data for embedding
item_embeddings = []
for itemID, info in item_info.items():
    # Combine relevant fields into a single text for embedding
    text = f"title: {info.get('title', '')}\nsalesRank: {info.get('salesRank', '')}\ndescription: {info.get('description', '')}"

    image = info.get('imUrl', '')

    item_embeddings.append({
        'ItemID': itemID,
        'text': text,
        'image': image,
        'price': info.get('price', 0),
        'brand': info.get('brand', 'NaN'),
        'categories': info.get('categories', []),
    })

# Convert to DataFrame
item_emb_df = pd.DataFrame(item_embeddings)

print("\nItem embeddings DataFrame shape:", item_emb_df.shape)
print("The first 3 rows of item embeddings DataFrame:\n", item_emb_df.head(3))


Item embeddings DataFrame shape: (12100, 6)
The first 3 rows of item embeddings DataFrame:
    ItemID                                               text  \
0       1  title: WAWO 15 Color Professionl Makeup Eyesha...   
1       2  title: Xtreme Brite Brightening Gel 1oz.\nsale...   
2       3  title: Prada Candy By Prada Eau De Parfum Spra...   

                                               image  price         brand  \
0  http://ecx.images-amazon.com/images/I/41Rn18Oe...   5.04          COKA   
1  http://ecx.images-amazon.com/images/I/41QWW9v1...  19.99  Xtreme Brite   
2  http://ecx.images-amazon.com/images/I/51iT2k6L...  65.86         Prada   

                                          categories  
0  [Beauty, Makeup, Face, Concealers & Neutralizers]  
1  [Beauty, Hair Care, Styling Products, Creams, ...  
2        [Beauty, Fragrance, Women's, Eau de Parfum]  


In [37]:
import pandas as pd
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests
from io import BytesIO
from tqdm import tqdm
import numpy as np
import os
import warnings

# 忽略不影响运行的警告
warnings.filterwarnings("ignore", category=UserWarning)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# ========== 1️⃣ 基础设置 ==========
dataset_name = "Beauty"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


# ========== 2️⃣ 定义辅助函数 ==========
def get_image_emb(url):
    """从URL获取图像embedding，失败则返回全0向量"""
    if not url or not isinstance(url, str) or not url.startswith("http"):
        return np.zeros(512)
    try:
        response = requests.get(url, timeout=5)
        img = Image.open(BytesIO(response.content)).convert("RGB")
        inputs = clip_processor(images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            emb = clip_model.get_image_features(**inputs)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        return emb.squeeze().cpu().numpy()
    except Exception:
        return np.zeros(512)


def get_text_emb(text):
    """获取文本embedding，自动截断到CLIP最大长度（77 tokens）"""
    if not text or not isinstance(text, str):
        return np.zeros(512)
    # 截断文本以防超过CLIP最大长度
    tokens = text.split()[:75]  # 预留 [CLS] 和 [EOS]
    text_short = " ".join(tokens)
    inputs = clip_processor(text=[text_short], return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        emb = clip_model.get_text_features(**inputs)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().cpu().numpy()


# ========== 3️⃣ 生成embedding ==========
text_embs, img_embs = [], []

print(f"Encoding {len(item_emb_df)} items with CLIP...")

for _, row in tqdm(item_emb_df.iterrows(), total=len(item_emb_df), desc="CLIP encoding"):
    text_emb = get_text_emb(row["text"]).tolist()
    img_emb = get_image_emb(row["image"]).tolist()
    text_embs.append(text_emb)
    img_embs.append(img_emb)

# ========== 4️⃣ 保存 ==========
item_emb_df_ = item_emb_df.copy()
item_emb_df_["text_emb"] = text_embs
item_emb_df_["image_emb"] = img_embs


Using device: cuda
Encoding 12100 items with CLIP...


CLIP encoding: 100%|██████████| 12100/12100 [40:40<00:00,  4.96it/s] 


In [38]:
# Save to parquet file
item_emb_df_.to_parquet(f'./{dataset_name}/item_emb.parquet', index=False)

print("Item embeddings saved to item_emb.parquet.")

Item embeddings saved to item_emb.parquet.


In [39]:
# 转成 numpy 数组
text_emb = np.stack(item_emb_df_['text_emb'].values)
image_emb = np.stack(item_emb_df_['image_emb'].values)

# 分别保存
np.save(f'./{dataset_name}/item_text_emb.npy', text_emb)
np.save(f'./{dataset_name}/item_image_emb.npy', image_emb)

print(f"✅ Saved item_text_emb.npy and item_image_emb.npy to ./{dataset_name}/")
print("text_emb shape:", text_emb.shape)
print("image_emb shape:", image_emb.shape)

✅ Saved item_text_emb.npy and item_image_emb.npy to ./Beauty/
text_emb shape: (12100, 512)
image_emb shape: (12100, 512)


In [42]:
pd.read_parquet(f'./{dataset_name}/item_emb.parquet')

,ItemID,text,image,price,brand,categories,text_emb,image_emb
0,1,title: WAWO 15 Color Professionl Makeup Eyesha...,http://ecx.images-amazon.com/images/I/41Rn18Oe...,5.04,COKA,"[Beauty, Makeup, Face, Concealers & Neutralizers]","[0.008328378200531006, -0.022224009037017822, ...","[-0.023478863760828972, 0.04270932823419571, 0..."
1,2,title: Xtreme Brite Brightening Gel 1oz.\nsale...,http://ecx.images-amazon.com/images/I/41QWW9v1...,19.99,Xtreme Brite,"[Beauty, Hair Care, Styling Products, Creams, ...","[0.00452251173555851, -0.009174905717372894, -...","[0.01667862758040428, 0.014809303916990757, 0...."
2,3,title: Prada Candy By Prada Eau De Parfum Spra...,http://ecx.images-amazon.com/images/I/51iT2k6L...,65.86,Prada,"[Beauty, Fragrance, Women's, Eau de Parfum]","[-0.04195597395300865, -0.07132028043270111, -...","[-0.030022917315363884, -0.04415012151002884, ..."
3,4,title: Versace Bright Crystal Eau de Toilette ...,http://ecx.images-amazon.com/images/I/418LYGLE...,52.33,Versace,"[Beauty, Fragrance, Women's, Eau de Toilette]","[0.007268585730344057, -0.04932981729507446, -...","[-0.01538288313895464, -0.042142171412706375, ..."
4,5,title: Stella McCartney Stella\nsalesRank: {'B...,http://ecx.images-amazon.com/images/I/31L2n60J...,NaN,None,"[Beauty, Fragrance, Women's, Eau de Parfum]","[-0.032064229249954224, -0.021080870181322098,...","[-0.04838688299059868, 0.021488051861524582, 0..."
...,...,...,...,...,...,...,...,...
12095,12097,title: Phytoceramides Anti Aging Supplement Re...,http://ecx.images-amazon.com/images/I/61enVb2X...,18.36,None,"[Beauty, Skin Care]","[0.04139937460422516, -0.013329751789569855, 0...","[-0.003335848217830062, 0.026186421513557434, ..."
12096,12096,"title: Moroccan Argan Oil - For Hair, Face, Sk...",http://ecx.images-amazon.com/images/I/41kTNm0k...,14.99,Natural Beauty,"[Beauty, Skin Care, Body, Moisturizers, Oils]","[-0.044768914580345154, -6.453951937146485e-05...","[-0.006330852396786213, 0.0036248036194592714,..."
12097,12098,title: LIME CRIME Velvetines - Wicked\nsalesRa...,http://ecx.images-amazon.com/images/I/41q7jpgt...,27.50,Lime Crime,"[Beauty, Makeup, Lips, Lipstick]","[0.007856694050133228, -0.0016975023318082094,...","[-0.04148480296134949, 0.011004832573235035, 0..."
12098,12099,title: Dr Song Rosehip Oil 4oz (4 oz)\nsalesRa...,http://ecx.images-amazon.com/images/I/412qdoPc...,19.99,None,"[Beauty, Skin Care, Face, Oils & Serums]","[-0.031039729714393616, 0.029431305825710297, ...","[0.007214525248855352, -0.02082887850701809, 0..."


In [49]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder

# =============================================================
# 复制，删除 text 和 image
item_emb_df2 = item_emb_df_.copy()
item_emb_df2.drop(columns=["text", "image"], inplace=True)

# =============================================================
# 1️⃣ 标准化价格
df_std = item_emb_df2.copy()
df_std["price"] = pd.to_numeric(df_std["price"], errors="coerce").fillna(0)
mean_price = df_std["price"].mean()
std_price = df_std["price"].std() if df_std["price"].std() != 0 else 1.0
df_std["price_norm"] = ((df_std["price"] - mean_price) / std_price).astype(np.float32)
df_std.drop(columns=["price"], inplace=True)

# =============================================================
# 2️⃣ 类别索引化：categories（多标签 → list[int]）
mlb = MultiLabelBinarizer()
cat_onehot = mlb.fit_transform(df_std["categories"])

# 转成索引列表
cat_idx_list = [np.where(row == 1)[0].astype(np.int64).tolist() for row in cat_onehot]
df_std["categories_idx"] = cat_idx_list
df_std.drop(columns=["categories"], inplace=True)

# =============================================================
# 3️⃣ 类别索引化：brand（单标签 → int）
brand_series = (
    df_std["brand"].copy()
    .replace(["NaN", "None", None], np.nan)
    .fillna("unknown")
)
brand_encoder = LabelEncoder()
brand_idx = brand_encoder.fit_transform(brand_series)
df_std["brand_idx"] = brand_idx.astype(np.int64)
df_std.drop(columns=["brand"], inplace=True)

# =============================================================
# 4️⃣ 整理字段
df_all = df_std.copy()

# ---- price_norm: 独立列（float32）
df_all["price_norm"] = df_all["price_norm"].apply(lambda x: np.array([x], dtype=np.float32))

# ---- brand_idx: 独立列（int64）
df_all["brand_idx"] = df_all["brand_idx"].apply(lambda x: np.array([x], dtype=np.int64))

# ---- categories_idx: 已经是 list[int]

# =============================================================
print("✅ 数据处理完毕，可直接送入 nn.Linear / nn.Embedding")
print("DataFrame columns:", df_all.columns.tolist())
print(df_all[["price_norm", "brand_idx", "categories_idx"]].head())

✅ 数据处理完毕，可直接送入 nn.Linear / nn.Embedding
DataFrame columns: ['ItemID', 'text_emb', 'image_emb', 'price_norm', 'categories_idx', 'brand_idx']
      price_norm brand_idx      categories_idx
0  [-0.56830996]     [300]   [17, 55, 88, 144]
1   [0.19413108]    [1973]  [17, 63, 105, 212]
2    [2.5334735]    [1456]   [17, 77, 99, 233]
3    [1.8434517]    [1896]   [17, 78, 99, 233]
4  [-0.82534695]    [2071]   [17, 77, 99, 233]


In [50]:
# Save to parquet file
df_all.to_parquet(f'./{dataset_name}/item_emb2.parquet', index=False)

print("Item embeddings saved to item_emb2.parquet.")

Item embeddings saved to item_emb2.parquet.


In [ ]:

#.tolist()

print("\nItem embeddings DataFrame shape:", item_emb_df.shape)
print("The first 3 rows of item embeddings DataFrame:\n", item_emb_df.head(3))

# Save to parquet file
item_emb_df.to_parquet(f'./{dataset_name}/item_emb.parquet', index=False)

print("Item embeddings saved to item_emb.parquet.")
# embeddings = np.array([item['embedding'] for item in item_embeddings])
# np.save(f'./{dataset_name}/item_emb.npy', embeddings)

# print("Item embeddings saved to item_emb.npy.")

In [51]:
import pandas as pd
data = pd.read_parquet('../data/Beauty/train.parquet')
print("max history id:", max(max(x) for x in data['history']))
print("max target id:", max(data['target']))

max history id: 12101
max target id: 12097
